# miRge3.0 — GSE114603
Dataset: `d_GSE114603/muestras_clasificadas` on Google Drive  
Run cells **top to bottom** on a fresh Colab session.

## 1. Environment setup (run once per session)

In [ ]:
!pip install mirge3 -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess

bowtie_bin = '/content/bowtie-1.3.0-linux-x86_64/bowtie-align-s'

# Try restoring from Drive first
import shutil
shutil.copytree(
    '/content/drive/MyDrive/miRge3.0/bowtie-1.3.0-linux-x86_64',
    '/content/bowtie-1.3.0-linux-x86_64',
    dirs_exist_ok=True
)
!chmod +x /content/bowtie-1.3.0-linux-x86_64/bowtie-align-s \
           /content/bowtie-1.3.0-linux-x86_64/bowtie-build-s \
           /content/bowtie-1.3.0-linux-x86_64/bowtie-inspect-s

# If still not executable, re-download
result = subprocess.run([bowtie_bin, '--version'], capture_output=True)
if result.returncode != 0:
    print('Drive copy not executable — re-downloading bowtie...')
    !wget -q -O /content/bowtie.zip https://sourceforge.net/projects/bowtie-bio/files/bowtie/1.3.0/bowtie-1.3.0-linux-x86_64.zip/download
    !unzip -q /content/bowtie.zip -d /content/
else:
    print('Drive copy OK')

# Create wrappers
tools = {
    'bowtie': 'bowtie-align-s',
    'bowtie-build': 'bowtie-build-s',
    'bowtie-inspect': 'bowtie-inspect-s',
}
for tool, binary in tools.items():
    wrapper = f'#!/bin/bash\nexec /content/bowtie-1.3.0-linux-x86_64/{binary} "$@"\n'
    p = f'/usr/local/bin/{tool}'
    with open(p, 'w') as f:
        f.write(wrapper)
    os.chmod(p, 0o755)
print('Bowtie ready')
!bowtie --version | head -1

In [ ]:
!apt-get install -y samtools -q

In [ ]:
!wget -q "https://www.tbi.univie.ac.at/RNA/download/sourcecode/2_4_x/ViennaRNA-2.4.16.tar.gz"
!tar -xzf ViennaRNA-2.4.16.tar.gz
!cd ViennaRNA-2.4.16 && ./configure --quiet && make -s && make install -s
!RNAfold --version

In [ ]:
%%bash
mkdir -p ~/mirge3_lib
cd ~/mirge3_lib
wget -q -O human.tar.gz "https://sourceforge.net/projects/mirge3/files/miRge3_Lib/human.tar.gz/download"
tar -xzf human.tar.gz
ls ~/mirge3_lib/

In [ ]:
# Patch miRgeEssential.py for Python 3.12 UTF-8 compatibility
filepath = '/usr/local/lib/python3.12/dist-packages/mirge/libs/miRgeEssential.py'
with open(filepath, 'r') as f:
    content = f.read()
content = content.replace(
    'capture_output=True, text=True)',
    'capture_output=True, text=True, encoding="utf-8", errors="replace")'
)
with open(filepath, 'w') as f:
    f.write(content)
print('Patched')

## 2. Run miRge3.0 on GSE114603

In [ ]:
from pathlib import Path

base = Path('/content/drive/MyDrive/miRge3.0/d_GSE114603/muestras_clasificadas')
fastq_files = sorted(base.rglob('*.fastq.gz'))
print(f'Found {len(fastq_files)} files:')
for f in fastq_files:
    print(f)

In [ ]:
import subprocess
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

base = Path('/content/drive/MyDrive/miRge3.0/d_GSE114603/muestras_clasificadas')
fastq_files = sorted(base.rglob('*.fastq.gz'))
output_base = Path('/content/output/GSE114603')

def run_mirge(fastq_path):
    sample = fastq_path.stem.replace('.fastq', '').replace('.fq', '')
    group = fastq_path.parent.name
    out_dir = output_base / group / sample
    out_dir.mkdir(parents=True, exist_ok=True)
    log_file = out_dir / 'mirge3.log'

    cmd = [
        'miRge3.0',
        '-s', str(fastq_path),
        '-lib', os.path.expanduser('~/mirge3_lib'),
        '-db', 'miRBase',
        '-on', 'human',
        '-ex', '0.1',
        '-ie',
        '-cpu', '1',
        '-o', str(out_dir),
        '-spl',
        '-a', 'illumina',
        '-nxt', '20',
        '-q', '20',
        '-NX',
        '-m', '18',
        '-mEC',
        '-gff'
    ]

    print(f'[START] {group}/{sample}')
    with open(log_file, 'w') as log:
        result = subprocess.run(cmd, stdout=log, stderr=log, cwd=str(out_dir))

    status = 'DONE' if result.returncode == 0 else f'FAILED (rc={result.returncode})'
    print(f'[{status}] {group}/{sample}')
    return sample, result.returncode

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = {executor.submit(run_mirge, f): f for f in fastq_files}
    for future in as_completed(futures):
        sample, code = future.result()

print('All samples processed!')

## 3. Save results to Drive

In [ ]:
import shutil
shutil.make_archive(
    '/content/drive/MyDrive/miRge3.0/d_GSE114603/mirge3_results',
    'zip',
    '/content/output/GSE114603'
)
print('Results saved to Drive!')